# AMEX Enterprise Credit Risk Platform
## Notebook 30 — Phase 2, Problem 3: Expected Credit Loss (IFRS9/CECL) — Business Understanding & ECL Policy
### Problem Statement 3 of 14 (depends on Problem 1's champion PD model and Problem 4's real tier-LGD)

CRISP-DM stage: **Business Understanding**. First of 4 notebooks for Problem 3 (Notebooks 30-33). Notebook 08 (Phase 1) already computed a worked Basel III / IFRS9 exercise using a single flat, explicitly-labeled `ASSUMPTION` LGD of 45% for every defaulter, and it explicitly flagged its own biggest shortcut: Stage 3 (credit-impaired) status was assigned using the real, already-known holdout outcome (`target=1`) — honest for a backtest illustration, but not how production staging works, since real IFRS9 staging must never depend on knowing the future.

**What Problem 3 genuinely improves, not just re-labels:**

- **Real tier-differentiated LGD, not flat.** Problem 4's Escalation Severity Score assigns each customer a real Low/Moderate/Severe tier from actual delinquency-trajectory features, each tier with its own LGD — replacing Notebook 08's single 45% for everyone.
- **Staging that never looks at the outcome.** Every stage in this notebook's rubric is assigned from signals a real institution would actually have *before* an account defaults: the customer's real predicted PD (Problem 1) and their real escalation severity tier (Problem 4). The known `target` label is used **only afterward, to validate** that the resulting stages rank-order with real observed default rates — never to assign a stage. This is the specific shortcut Notebook 08 flagged; this notebook does not take it.
- **IFRS 9 vs CECL side by side.** IFRS 9 requires 3-stage ECL (12-month for Stage 1, lifetime for Stage 2/3); US GAAP CECL requires lifetime expected credit loss for the *entire* portfolio from day one, with no staging at all. Notebook 31 computes both from the same real inputs and shows the real dollar gap between the two standards.
- **Forward-looking macro overlay and discounting**, both explicit IFRS9/CECL requirements Notebook 08 didn't attempt.

**Honesty boundary, stated up front:** this is still a single-snapshot Kaggle dataset with no PD term structure and no real macroeconomic data feed. The staging rubric, lifetime-PD approximation, discount rate, and macro scenario weights below are explicit, editable `ASSUMPTION`s — clearly separated from the real, live-measured PD and tier inputs that feed them. That is the same honesty convention every notebook in this platform follows.

**Deliverables:** `ecl_policy.json` (read by every later Problem 3 notebook), `p3_stakeholder_analysis.csv`, `ECL_Policy_Charter.docx`.

**Run the single code cell below, once.** Idempotent — every output file is overwritten in place on every re-run.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD PROBLEM 1's AND PROBLEM 4's REAL RESULTS
# =============================================================================
import os
import sys
import json
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Problem 1's and Problem 4's Real Results")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
P1_ARTIFACTS = PROJECT_ROOT / "Phase1_Foundation" / "Problem1_Credit_Scoring_PD_Prediction" / "artifacts"
P4_ROOT = PROJECT_ROOT / "Phase2_Regulatory_Loss_Provisioning" / "Problem4_Delinquency_Escalation_Loss_Severity"
P4_ARTIFACTS = P4_ROOT / "artifacts"
ARTIFACTS_DIR = PROJECT_ROOT / "Phase2_Regulatory_Loss_Provisioning" / "Problem3_Expected_Credit_Loss_IFRS9_CECL" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

P1_CONFIG_PATH = P1_ARTIFACTS / "project_config.json"
NB05_SUMMARY_PATH = P1_ARTIFACTS / "notebook_05_summary.json"
NB08_SUMMARY_PATH = P1_ARTIFACTS / "notebook_08_summary.json"
P4_LGD_POLICY_PATH = P4_ROOT / "01_LGD_Policy" / "lgd_policy.json"
NB27_SUMMARY_PATH = P4_ARTIFACTS / "notebook_27_summary.json"
NB28_SUMMARY_PATH = P4_ARTIFACTS / "notebook_28_summary.json"
for _p, _fix in [
    (P1_CONFIG_PATH, "run Problem 1's Notebook 01 first."),
    (NB05_SUMMARY_PATH, "Problem 3 needs Problem 1's champion PD model -- run Problem 1's Notebook 05 first."),
    (NB08_SUMMARY_PATH, "Problem 3 compares against Notebook 08's flat-LGD ECL -- run Problem 1's Notebook 08 first."),
    (P4_LGD_POLICY_PATH, "Problem 3 needs Problem 4's tier-LGD policy -- run Problem 4's Notebook 26 first."),
    (NB27_SUMMARY_PATH, "Problem 3 needs Problem 4's validated severity tiers -- run Problem 4's Notebook 27 first."),
    (NB28_SUMMARY_PATH, "Problem 3 needs Problem 4's deployment validation -- run Problem 4's Notebook 28 first."),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix}")

with open(P1_CONFIG_PATH, "r", encoding="utf-8") as f:
    P1_CONFIG = json.load(f)
with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB05_SUMMARY = json.load(f)
with open(NB08_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB08_SUMMARY = json.load(f)
with open(P4_LGD_POLICY_PATH, "r", encoding="utf-8") as f:
    P4_LGD_POLICY = json.load(f)
with open(NB27_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB27_SUMMARY = json.load(f)
with open(NB28_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB28_SUMMARY = json.load(f)

if not NB28_SUMMARY.get("self_test_passed", False):
    raise RuntimeError(
        "Problem 4's Notebook 28 self-test did not pass on its last real run -- Problem 3 depends on "
        "Problem 4's tier-LGD being deployment-verified before building on it.\n"
        "Fix: re-run Problem 4's Notebook 27 then Notebook 28 until the self-test passes, then re-run this notebook."
    )

RANDOM_SEED = P1_CONFIG["random_seed"]
_resource_limits = P1_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (
    _resource_limits.get("warp_thread_count")
    or P1_CONFIG.get("warp_thread_count")
    or P1_CONFIG["hardware"]["logical_cores_detected"]
)

CHAMPION_NAME = NB05_SUMMARY["champion_model"]
BASELINE_LGD_FLAT = NB08_SUMMARY["lgd_assumption"]              # Notebook 08's real flat-LGD value
BASELINE_EAD_USD = NB08_SUMMARY["ead_per_account_usd_assumption"]
NB08_TOTAL_ECL_USD = NB08_SUMMARY["total_ecl_usd"]               # Notebook 08's real comparison-baseline ECL
P4_TIER_ORDER = P4_LGD_POLICY["tier_order"]                      # real, from Problem 4
P4_LGD_BY_TIER = {v["tier"]: v["lgd"] for v in P4_LGD_POLICY["lgd_by_tier"]["values"]}  # real, from Problem 4
P4_EAD_USD = P4_LGD_POLICY["ead_per_account_usd"]
P4_MONOTONIC_VALIDATED = NB27_SUMMARY["monotonic_validated"]
P4_SEVERE_TO_LOW_RATIO = NB27_SUMMARY["severe_to_low_default_rate_ratio"]
P4_SELF_TEST_PASSED = NB28_SUMMARY["self_test_passed"]

print(f"Problem 1 champion model (real)              : {CHAMPION_NAME}")
print(f"Notebook 08 baseline flat LGD (real)          : {BASELINE_LGD_FLAT:.0%}")
print(f"Notebook 08 baseline EAD/account (real)        : ${BASELINE_EAD_USD:,}")
print(f"Notebook 08 total ECL, flat LGD (real, holdout): ${NB08_TOTAL_ECL_USD:,.0f}")
print(f"Problem 4 severity tiers (real)                : {P4_TIER_ORDER}")
print(f"Problem 4 LGD by tier (real, ASSUMPTION dollars): {P4_LGD_BY_TIER}")
print(f"Problem 4 tiers rank-order monotonic (real)    : {P4_MONOTONIC_VALIDATED}")
print(f"Problem 4 Severe/Low default-rate ratio (real) : {P4_SEVERE_TO_LOW_RATIO:.2f}x")
print(f"Problem 4 deployment self-test passed (real)   : {P4_SELF_TEST_PASSED}")

# --- This problem's own folder skeleton (scaffolded already; self-heal here too) ---
P3_ROOT = PROJECT_ROOT / "Phase2_Regulatory_Loss_Provisioning" / "Problem3_Expected_Credit_Loss_IFRS9_CECL"
PILLAR_DIRS = {
    "p3_policy": P3_ROOT / "01_ECL_Policy",
    "p3_modeling": P3_ROOT / "02_ECL_Modeling",
    "p3_validation_deployment": P3_ROOT / "03_Validation_Deployment",
    "p3_reporting_packaging": P3_ROOT / "04_Financial_Impact_Reporting_Packaging",
}
for _d in PILLAR_DIRS.values():
    _d.mkdir(parents=True, exist_ok=True)

# --- This problem's own project_config.json (mirrors Problem 1's real hardware
#     facts -- no re-detection needed, the machine hasn't changed -- plus this
#     problem's own pillar_dirs and data_root). Self-healed/overwritten each run. ---
P3_CONFIG = {
    "project_name": "AMEX Enterprise Credit Risk Platform",
    "config_generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "project_root": str(PROJECT_ROOT),
    "data_root": P1_CONFIG["data_root"],
    "problem_root": str(P3_ROOT),
    "artifacts_dir": str(ARTIFACTS_DIR),
    "random_seed": RANDOM_SEED,
    "hardware": P1_CONFIG["hardware"],
    "resource_limits": P1_CONFIG.get("resource_limits", {}),
    "warp_thread_count": WARP_THREAD_COUNT,
    "pillar_dirs": {k: str(v) for k, v in PILLAR_DIRS.items()},
}
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(P3_CONFIG, f, indent=2)

print(f"\n\u2705 project_config.json written -- Notebooks 31-33 load this file.")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: Library Imports")

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

missing = []
try:
    import pandas as pd
except ImportError:
    missing.append("pandas")
try:
    from docx import Document
except ImportError:
    missing.append("python-docx")
if missing:
    raise ImportError("Missing required package(s): " + ", ".join(missing) +
                       "\nFix: pip install " + " ".join(missing))

print("(Reporting only -- this notebook defines policy; it does no data-scale work.)")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: PROBLEM 3 -- BUSINESS PROBLEM & KPI TREE
# =============================================================================
_section("SECTION 3: Problem 3 -- Business Problem & KPI Tree")

PROBLEM_3_CONTEXT = {
    "problem_name": "Expected Credit Loss (IFRS9/CECL)",
    "problem_number": 3,
    "phase": "Phase 2 -- Regulatory & Loss Provisioning",
    "depends_on": [
        "Problem 1: Credit Scoring / PD Prediction (Notebooks 01-18) -- real champion PD model",
        "Problem 4: Delinquency Escalation / Loss Severity (Notebooks 26-29) -- real tier-differentiated LGD",
    ],
    "problem_statement": (
        f"Notebook 08 computes Expected Credit Loss with a single flat {BASELINE_LGD_FLAT:.0%} LGD for every "
        "defaulter and assigns IFRS9 Stage 3 using the real, already-known holdout outcome -- both explicitly "
        "flagged there as shortcuts. Problem 3 replaces the flat LGD with Problem 4's real tier-differentiated "
        "LGD, and replaces outcome-based staging with a rubric built entirely from signals available BEFORE "
        "default (real PD, real severity tier) -- the known outcome is used only afterward, to validate that "
        "result, never to assign it. Problem 3 also computes both IFRS9 (staged) and CECL (lifetime, "
        "unstaged) ECL from the same real inputs, and adds the forward-looking macro overlay and discounting "
        "both standards require."
    ),
    "kpi_tree": [
        "Outcome-free staging: every stage assignment must be computable from information available before "
        "default (PD, severity tier) -- the real observed default label is used only to VALIDATE staging "
        "afterward, never to assign it.",
        "Rank-ordering / monotonicity: real observed default rate must increase strictly from Stage 1 to "
        "Stage 3 -- a non-monotonic scheme cannot justify differentiated provisioning.",
        "Methodological transparency: every dollar in the final ECL must trace to either a real, live-measured "
        "input (PD, tier, population counts) or an explicit, separately-labeled ASSUMPTION (LGD dollars, EAD, "
        "lifetime-PD approximation, discount rate, macro scenario weights).",
        "Standard comparability: IFRS9 (staged) and CECL (lifetime-for-all) ECL must both be computed from "
        "the identical real inputs, so the reported gap between them reflects the accounting standard, not a "
        "difference in underlying data.",
    ],
}
print(json.dumps(PROBLEM_3_CONTEXT, indent=2))
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: ECL POLICY -- THE SINGLE SOURCE OF TRUTH FOR NOTEBOOKS 31-33
# =============================================================================
_section("SECTION 4: ECL Policy -- The Single Source of Truth for Notebooks 31-33")

# --- Staging thresholds, lifetime-PD approximation, discount rate, and macro
#     scenario weights are explicit ASSUMPTIONs -- this dataset has no PD term
#     structure and no real macro data feed. LGD by tier and EAD are NOT
#     re-invented here -- they are read programmatically from Problem 4's real
#     policy above, so this file can never silently drift out of sync with it. ---
ECL_POLICY = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "framework": "IFRS 9 (3-stage: 12-month ECL for Stage 1, lifetime ECL for Stage 2/3) computed side by "
                 "side with CECL (lifetime ECL for 100% of the portfolio from day one, no staging) -- Notebook "
                 "31 computes both from the identical real PD/tier inputs.",
    "staging_criteria": {
        "description": "Outcome-free by design: every criterion below uses only signals a real institution "
                        "has BEFORE a default happens (real PD from Problem 1, real escalation severity tier "
                        "from Problem 4). The real observed default label is used later, in Notebook 31, only "
                        "to VALIDATE that the resulting stages rank-order with real outcomes -- never to "
                        "decide a stage. This is the specific shortcut Notebook 08 took and flagged (its "
                        "Stage 3 used the real known outcome directly) -- this policy does not take it.",
        "stage_1_performing": "Not Stage 2 or Stage 3 (see below).",
        "stage_2_sicr": "PD > sicr_pd_multiple (below) times the portfolio-average PD, OR severity tier is "
                         "Moderate or Severe (Problem 4). Either condition alone is a real, pre-default signal "
                         "of significant increase in credit risk.",
        "stage_3_credit_impaired": "Severity tier is Severe AND PD > stage3_pd_threshold (below) -- an "
                                    "absolute, not relative, threshold; both conditions together are a "
                                    "defensible outcome-free proxy for objective evidence of impairment.",
        "sicr_pd_multiple": 3.0,               # ASSUMPTION -- same convention Notebook 08 used for its own SICR proxy
        "stage3_pd_threshold": 0.50,            # ASSUMPTION -- real predicted PD above 50% AND already worst tier
    },
    "lifetime_pd": {
        "description": "ASSUMPTION -- lifetime PD approximated as min(1, PD_12m * lifetime_pd_multiplier), "
                        "the same illustrative approach Notebook 08 used (no PD term-structure data exists in "
                        "this single-snapshot dataset to fit a real survival curve).",
        "lifetime_pd_multiplier": 3.0,          # ASSUMPTION -- matches Notebook 08's IFRS9_LIFETIME_PD_MULTIPLIER
    },
    "lgd_by_tier": {
        "description": "Real values, read programmatically from Problem 4's lgd_policy.json -- not re-typed "
                        "or re-invented here.",
        "tier_order": P4_TIER_ORDER,
        "values": P4_LGD_BY_TIER,
        "source": "Problem 4, Notebook 26 (real value, read programmatically)",
    },
    "ead_per_account_usd": P4_EAD_USD,   # ASSUMPTION, inherited unchanged from Problem 4 (itself from Notebook 08)
    "discount_rate": {
        "description": "ASSUMPTION -- effective interest rate (EIR) used to discount expected cash shortfalls "
                        "to present value (IFRS9 \u00a75.5.17); dataset has no real EIR, so a representative "
                        "unsecured-revolving-credit annual rate is used. Discounting needs a TIME as well as a "
                        "rate -- Stage 1's 12-month loss is discounted from the mid-point of the 12-month "
                        "collection window (a standard simplifying convention); Stage 2/3's lifetime loss is "
                        "discounted from an average-remaining-life ASSUMPTION, since this dataset has no real "
                        "account-tenure data to compute one.",
        "annual_rate": 0.199,                        # ASSUMPTION -- ~19.9% typical US unsecured revolving-credit APR
        "stage1_discount_period_years": 0.5,          # ASSUMPTION -- mid-point-of-year convention
        "stage23_avg_remaining_life_years": 2.0,      # ASSUMPTION -- no real tenure data to derive this from
    },
    "macro_overlay": {
        "description": "ASSUMPTION -- forward-looking macroeconomic scenario weighting, required by both "
                        "IFRS9 \u00a75.5.17(c) and CECL's forward-looking-information requirement. No real "
                        "macro data feed exists for this dataset, so probability-weighted PD multipliers "
                        "stand in for a real macro model's output -- editable to one directly.",
        "scenarios": [
            {"scenario": "Upside", "probability": 0.20, "pd_multiplier": 0.85},
            {"scenario": "Baseline", "probability": 0.50, "pd_multiplier": 1.00},
            {"scenario": "Downside", "probability": 0.30, "pd_multiplier": 1.35},
        ],
    },
    "kpi_targets": {
        "require_strict_monotonicity_default_rate_by_stage": True,   # ASSUMPTION
        "min_stage3_population_pct": 0.5,     # ASSUMPTION -- sanity floor, not a target to hit
        "max_stage3_population_pct": 25.0,    # ASSUMPTION -- sanity ceiling, not a target to hit
    },
    "champion_pd_model_used": CHAMPION_NAME,
    "flat_lgd_comparison_baseline": {
        "source": "Problem 1, Notebook 08 (real value, read programmatically)",
        "lgd_flat": BASELINE_LGD_FLAT, "total_ecl_usd": NB08_TOTAL_ECL_USD,
    },
}

_macro_prob_sum = sum(s["probability"] for s in ECL_POLICY["macro_overlay"]["scenarios"])
if abs(_macro_prob_sum - 1.0) > 1e-9:
    raise RuntimeError(f"Macro scenario probabilities must sum to 1.0, got {_macro_prob_sum}. Fix the "
                        "ECL_POLICY['macro_overlay']['scenarios'] block above.")

ecl_policy_path = PILLAR_DIRS["p3_policy"] / "ecl_policy.json"
with open(ecl_policy_path, "w", encoding="utf-8") as f:
    json.dump(ECL_POLICY, f, indent=2)

print(json.dumps(ECL_POLICY, indent=2))
print(f"\n\u2705 Saved -> {ecl_policy_path}")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: STAKEHOLDER ANALYSIS
# =============================================================================
_section("SECTION 5: Stakeholder Analysis")

STAKEHOLDERS = [
    {"stakeholder": "Finance / Provisioning", "interest": "The IFRS9 vs CECL ECL gap directly changes the "
     "quarterly reserve booked under each standard -- needs a defensible, auditable bridge between the two."},
    {"stakeholder": "Regulatory Reporting / Basel Desk", "interest": "Needs Notebook 08's flat-LGD Basel "
     "capital figures reconciled against this notebook's tier-LGD ECL -- both must trace to the same real PD."},
    {"stakeholder": "Model Risk / Compliance (SR 11-7)", "interest": "Needs the staging rubric documented as "
     "outcome-free (no reliance on the known label) before this feeds any regulatory-adjacent provisioning "
     "calculation -- exactly the gap this notebook closes versus Notebook 08."},
    {"stakeholder": "Executive / CFO", "interest": "Needs the dollar impact of moving from Notebook 08's flat "
     "LGD to Problem 4's tier-differentiated LGD, and the IFRS9-vs-CECL gap, stated in plain financial terms "
     "(Notebook 33)."},
]
stakeholder_df = pd.DataFrame(STAKEHOLDERS)
stakeholder_path = PILLAR_DIRS["p3_policy"] / "p3_stakeholder_analysis.csv"
stakeholder_df.to_csv(stakeholder_path, index=False)
print(stakeholder_df.to_string(index=False))
print(f"\u2705 Saved -> {stakeholder_path}")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: WORD REPORT -- ECL_POLICY_CHARTER.DOCX
# =============================================================================
_section("SECTION 6: Word Report -- ECL_Policy_Charter.docx")


def _add_heading(doc, text, level=1):
    return doc.add_heading(text, level=level)


def _add_kv_table(doc, data: dict):
    table = doc.add_table(rows=0, cols=2)
    table.style = "Light Grid Accent 1"
    for k, v in data.items():
        row = table.add_row().cells
        row[0].text = str(k).replace("_", " ").title()
        row[1].text = "" if v is None else str(v)
    return table


def _add_table_from_df(doc, df, max_rows=30):
    table = doc.add_table(rows=1, cols=len(df.columns))
    table.style = "Light Grid Accent 1"
    hdr = table.rows[0].cells
    for i, col in enumerate(df.columns):
        hdr[i].text = str(col).replace("_", " ").title()
    for _, row in df.head(max_rows).iterrows():
        cells_ = table.add_row().cells
        for i, col in enumerate(df.columns):
            cells_[i].text = "" if pd.isna(row[col]) else str(row[col])
    return table


doc = Document()
doc.add_heading("AMEX Enterprise Credit Risk Platform", level=0)
doc.add_paragraph("Phase 2, Problem 3: Expected Credit Loss (IFRS9/CECL) -- Business Understanding & ECL Policy Charter")
doc.add_paragraph(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

_add_heading(doc, "1. Problem Statement", level=1)
doc.add_paragraph(PROBLEM_3_CONTEXT["problem_statement"])

_add_heading(doc, "2. KPI Tree", level=1)
for _k in PROBLEM_3_CONTEXT["kpi_tree"]:
    doc.add_paragraph(_k, style="List Bullet")

_add_heading(doc, "3. Staging Criteria (outcome-free by design)", level=1)
doc.add_paragraph(ECL_POLICY["staging_criteria"]["description"])
_add_kv_table(doc, {
    "Stage 1 (Performing)": ECL_POLICY["staging_criteria"]["stage_1_performing"],
    "Stage 2 (SICR)": ECL_POLICY["staging_criteria"]["stage_2_sicr"],
    "Stage 3 (Credit-Impaired)": ECL_POLICY["staging_criteria"]["stage_3_credit_impaired"],
    "SICR PD Multiple (ASSUMPTION)": ECL_POLICY["staging_criteria"]["sicr_pd_multiple"],
    "Stage 3 PD Threshold (ASSUMPTION)": ECL_POLICY["staging_criteria"]["stage3_pd_threshold"],
})

_add_heading(doc, "4. LGD by Tier (real, from Problem 4)", level=1)
doc.add_paragraph(ECL_POLICY["lgd_by_tier"]["description"])
_add_table_from_df(doc, pd.DataFrame(
    [{"tier": t, "lgd": ECL_POLICY["lgd_by_tier"]["values"][t]} for t in ECL_POLICY["lgd_by_tier"]["tier_order"]]))

_add_heading(doc, "5. Lifetime PD, Discount Rate & Macro Overlay (ASSUMPTION)", level=1)
doc.add_paragraph(ECL_POLICY["lifetime_pd"]["description"])
doc.add_paragraph(ECL_POLICY["discount_rate"]["description"])
doc.add_paragraph(ECL_POLICY["macro_overlay"]["description"])
_add_table_from_df(doc, pd.DataFrame(ECL_POLICY["macro_overlay"]["scenarios"]))

_add_heading(doc, "6. KPI Targets (ASSUMPTION)", level=1)
_add_kv_table(doc, ECL_POLICY["kpi_targets"])

_add_heading(doc, "7. Stakeholder Analysis", level=1)
_add_table_from_df(doc, stakeholder_df)

report_path = PILLAR_DIRS["p3_policy"] / "ECL_Policy_Charter.docx"
doc.save(str(report_path))
print(f"\u2705 Saved -> {report_path}")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: VERIFICATION
# =============================================================================
_section("SECTION 7: Verification")

_checks_passed = True


def _check(label, condition, detail=""):
    global _checks_passed
    if condition:
        print(f"\u2705 {label}")
    else:
        _checks_passed = False
        print(f"\u274c {label}  {detail}")


_check("Macro scenario probabilities sum to 1.0", abs(_macro_prob_sum - 1.0) < 1e-9)
_check("LGD by tier matches Problem 4's real policy exactly",
       ECL_POLICY["lgd_by_tier"]["values"] == P4_LGD_BY_TIER)
_check("EAD/account inherited unchanged from Problem 4", ECL_POLICY["ead_per_account_usd"] == P4_EAD_USD)
_check("Stage 3 PD threshold is a real probability (0-1)",
       0.0 < ECL_POLICY["staging_criteria"]["stage3_pd_threshold"] < 1.0)
_check("Lifetime PD multiplier is >= 1.0 (lifetime PD cannot be below 12-month PD)",
       ECL_POLICY["lifetime_pd"]["lifetime_pd_multiplier"] >= 1.0)

_expected_files = [ecl_policy_path, stakeholder_path, report_path]
for fp in _expected_files:
    _check(f"{fp.name} exists and is non-empty", fp.exists() and fp.stat().st_size > 0)

if not _checks_passed:
    raise RuntimeError("One or more Notebook 30 verification checks failed. See \u274c lines above.")

print("\nAll Notebook 30 checks passed.")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: WRITE NOTEBOOK 30 SUMMARY ARTIFACT
# =============================================================================
_section("SECTION 8: Write Notebook 30 Summary Artifact")

notebook_30_summary = {
    "notebook": "30_ecl_policy", "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem_number": 3, "problem_name": "Expected Credit Loss (IFRS9/CECL)",
    "phase": "Phase 2 -- Regulatory & Loss Provisioning",
    "champion_pd_model": CHAMPION_NAME, "lgd_by_tier": P4_LGD_BY_TIER, "ead_per_account_usd": P4_EAD_USD,
    "flat_lgd_comparison_baseline_usd": NB08_TOTAL_ECL_USD,
    "output_files": {p.name: str(p) for p in _expected_files},
}
nb30_summary_path = ARTIFACTS_DIR / "notebook_30_summary.json"
with open(nb30_summary_path, "w", encoding="utf-8") as f:
    json.dump(notebook_30_summary, f, indent=2)
print(f"\u2705 Saved -> {nb30_summary_path}")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: COMPLETION SUMMARY
# =============================================================================
_section("SECTION 9: Notebook 30 Complete -- Handoff to Notebook 31")

print("NOTEBOOK 30: EXPECTED CREDIT LOSS (IFRS9/CECL) -- BUSINESS UNDERSTANDING & ECL POLICY -- COMPLETE")
print(f"  Champion PD model (Problem 1, real) : {CHAMPION_NAME}")
print(f"  LGD by tier (Problem 4, real)       : {P4_LGD_BY_TIER}")
print(f"  Staging rubric                      : outcome-free (PD + severity tier only)")
print(f"  Files produced                      : {len(_expected_files) + 1}")
for _p in _expected_files + [nb30_summary_path]:
    print(f"    - {_p.name}")
print(f"  Next notebook                       : 31_ecl_modeling.ipynb")
print("\n\u2705 Ready to proceed.")
